<a href="https://colab.research.google.com/github/rahafabumwise/IEEE-AI-Modeling-Hackathon-2.0-Stage-1-Challenge/blob/main/clean_best_model_validation_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clean Best Model + Validation Lab

This notebook keeps only the parts that matter:

- the exact 58-feature thermal/stress pipeline behind the **0.20500 Kaggle submission**
- fold-safe 5-fold OOF validation
- CatBoost, LightGBM, and fixed 80/20 blend checkpoints
- a fixed **test-like validation checkpoint** based on adversarial validation
- saved OOF predictions and a compact experiment scoreboard
- exact full-data training and submission generation

Rejected experiments such as GMM regime probabilities, regime interactions, density weighting, manual corrections, and repeated residual binning are intentionally excluded.

## Workflow

1. Run setup and load data.
2. Run the shift checkpoint once.
3. Run the exact baseline OOF section.
4. Record the baseline checkpoint.
5. Test one controlled candidate at a time.
6. Generate a submission only after a candidate passes the checkpoints.


In [1]:
!pip install -q catboost lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.2 MB/s eta 0:00:00


In [2]:
import os
import json
import numpy as np
import pandas as pd
import lightgbm as lgb

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor, LGBMClassifier

from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import mean_squared_log_error, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 200)


## 1. Load data

Change only `DATA_PATH` when needed.

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
DATA_PATH = "/content/drive/MyDrive/datasets/ieee-ai-modeling-hackathon2-stage-1-challenge"

train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

X_raw = train.drop(columns=["id", "edi"]).copy()
X_test_raw = test.drop(columns=["id"]).copy()

y_raw = train["edi"].copy()
y_log = np.log1p(y_raw)

print("Train:", train.shape)
print("Test:", test.shape)
print("Raw train features:", X_raw.shape)
print("Raw test features:", X_test_raw.shape)

assert train.shape == (24000, 44)
assert test.shape == (16000, 43)
assert X_raw.shape[1] == 42
assert list(X_raw.columns) == list(X_test_raw.columns)


Train: (24000, 44)
Test: (16000, 43)
Raw train features: (24000, 42)
Raw test features: (16000, 42)


## 2. Exact preprocessing and feature engineering

All learned preprocessing is fitted on the training fold only.

In [5]:
MISSING_COLS = [
    "humidity",
    "sensor_17",
    "vibration_rms",
    "coolant_flow",
    "hours_since_service",
    "sensor_05",
]


def preprocess_fold(X_train_fold, X_valid_fold):
    X_train_fold = X_train_fold.copy()
    X_valid_fold = X_valid_fold.copy()

    for col in MISSING_COLS:
        X_train_fold[f"{col}_was_missing"] = (
            X_train_fold[col].isna().astype(np.int8)
        )
        X_valid_fold[f"{col}_was_missing"] = (
            X_valid_fold[col].isna().astype(np.int8)
        )

    X_train_fold["total_missing_count"] = (
        X_train_fold[MISSING_COLS].isna().sum(axis=1)
    )
    X_valid_fold["total_missing_count"] = (
        X_valid_fold[MISSING_COLS].isna().sum(axis=1)
    )

    fold_medians = {}
    for col in MISSING_COLS:
        median_value = X_train_fold[col].median()
        fold_medians[col] = float(median_value)

        X_train_fold[col] = X_train_fold[col].fillna(median_value)
        X_valid_fold[col] = X_valid_fold[col].fillna(median_value)

    return X_train_fold, X_valid_fold, fold_medians


In [6]:
def add_engineered_features_stress(X):
    X = X.copy()

    X["load_duty_combo"] = (
        X["load_factor"] * X["duty_cycle"]
    )

    X["stress_index"] = (
        X["core_temp"]
        * X["load_factor"]
        * X["duty_cycle"]
    )

    X["thermal_excess"] = X["delta_ambient"]

    X["electrical_stress"] = (
        X["harmonic_thd"] * X["load_factor"]
    )

    X["mechanical_stress"] = (
        X["vibration_rms"] * X["load_factor"]
    )

    X["combined_operating_stress"] = (
        X["core_temp"]
        * X["load_factor"]
        * X["duty_cycle"]
        * (1.0 + X["harmonic_thd"])
    )

    return X


In [7]:
TEMP_PREDICTOR_FEATURES = [
    "asset_age",
    "load_factor",
    "duty_cycle",
    "coolant_flow",
    "humidity",
    "vibration_rms",
    "grid_freq",
    "line_voltage",
    "harmonic_thd",
    "phase_imbalance",
    "hours_since_service",
    "chamber_pressure",
]


def add_temperature_residual_features(X_train_raw, X_valid_raw):
    X_train_raw = X_train_raw.copy()
    X_valid_raw = X_valid_raw.copy()

    temperature_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0)),
    ])

    temperature_model.fit(
        X_train_raw[TEMP_PREDICTOR_FEATURES],
        X_train_raw["core_temp"],
    )

    train_expected = temperature_model.predict(
        X_train_raw[TEMP_PREDICTOR_FEATURES]
    )
    valid_expected = temperature_model.predict(
        X_valid_raw[TEMP_PREDICTOR_FEATURES]
    )

    X_train_raw["expected_core_temp"] = train_expected
    X_valid_raw["expected_core_temp"] = valid_expected

    X_train_raw["core_temp_residual"] = (
        X_train_raw["core_temp"] - train_expected
    )
    X_valid_raw["core_temp_residual"] = (
        X_valid_raw["core_temp"] - valid_expected
    )

    X_train_raw["abs_core_temp_residual"] = (
        X_train_raw["core_temp_residual"].abs()
    )
    X_valid_raw["abs_core_temp_residual"] = (
        X_valid_raw["core_temp_residual"].abs()
    )

    return X_train_raw, X_valid_raw, temperature_model


In [8]:
def calculate_rmsle(actual, prediction):
    actual = np.asarray(actual)
    prediction = np.clip(np.asarray(prediction), 0, None)

    return float(
        np.sqrt(mean_squared_log_error(actual, prediction))
    )


def build_fold_features(X_train_raw, X_valid_raw):
    X_train_thermal, X_valid_thermal, temperature_model = (
        add_temperature_residual_features(
            X_train_raw,
            X_valid_raw,
        )
    )

    X_train, X_valid, medians = preprocess_fold(
        X_train_thermal,
        X_valid_thermal,
    )

    X_train = add_engineered_features_stress(X_train)
    X_valid = add_engineered_features_stress(X_valid)

    assert list(X_train.columns) == list(X_valid.columns)
    assert X_train.isna().sum().sum() == 0
    assert X_valid.isna().sum().sum() == 0
    assert X_train.shape[1] == 58
    assert X_valid.shape[1] == 58

    return X_train, X_valid, temperature_model, medians


## 3. Shift checkpoint: how different are train and test?

This section creates a cross-fitted `test_likeness_score` for every training row.

- AUC near 0.50 means train and test are difficult to distinguish.
- AUC clearly above 0.50 confirms distribution shift.
- The top 20% most test-like training rows form a fixed secondary validation checkpoint.

This checkpoint does not replace normal OOF. It complements it.

In [9]:
combined_X = pd.concat(
    [X_raw, X_test_raw],
    axis=0,
    ignore_index=True,
)

combined_origin = np.concatenate([
    np.zeros(len(X_raw), dtype=np.int8),
    np.ones(len(X_test_raw), dtype=np.int8),
])

shift_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

origin_oof_probability = np.zeros(len(combined_X))

for fold, (fit_idx, valid_idx) in enumerate(
    shift_cv.split(combined_X, combined_origin),
    start=1,
):
    shift_model = LGBMClassifier(
        objective="binary",
        n_estimators=1200,
        learning_rate=0.03,
        num_leaves=31,
        min_child_samples=30,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=SEED + fold,
        n_jobs=-1,
        verbosity=-1,
    )

    shift_model.fit(
        combined_X.iloc[fit_idx],
        combined_origin[fit_idx],
        eval_set=[(
            combined_X.iloc[valid_idx],
            combined_origin[valid_idx],
        )],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    origin_oof_probability[valid_idx] = shift_model.predict_proba(
        combined_X.iloc[valid_idx],
        num_iteration=shift_model.best_iteration_,
    )[:, 1]

shift_auc = roc_auc_score(
    combined_origin,
    origin_oof_probability,
)

train_test_likeness = origin_oof_probability[:len(X_raw)]

test_like_threshold = np.quantile(
    train_test_likeness,
    0.80,
)

very_test_like_mask = (
    train_test_likeness >= test_like_threshold
)

print("Adversarial validation AUC:", shift_auc)
print("Test-like train rows:", int(very_test_like_mask.sum()))
print(pd.Series(train_test_likeness).describe())


Adversarial validation AUC: 0.6398411640625
Test-like train rows: 4800
count    24000.000000
mean         0.376958
std          0.109836
min          0.139532
25%          0.292533
50%          0.364865
75%          0.449434
max          0.789407
dtype: float64


In [10]:
# Simple univariate shift report for interpretation only
from scipy.stats import ks_2samp

shift_rows = []

for col in X_raw.columns:
    train_values = X_raw[col].dropna()
    test_values = X_test_raw[col].dropna()

    ks_stat, p_value = ks_2samp(
        train_values,
        test_values,
    )

    shift_rows.append({
        "feature": col,
        "ks_stat": ks_stat,
        "p_value": p_value,
        "train_mean": train_values.mean(),
        "test_mean": test_values.mean(),
        "train_missing_pct": X_raw[col].isna().mean() * 100,
        "test_missing_pct": X_test_raw[col].isna().mean() * 100,
    })

shift_report = (
    pd.DataFrame(shift_rows)
    .sort_values("ks_stat", ascending=False)
    .reset_index(drop=True)
)

display(shift_report.head(15))


,feature,ks_stat,p_value,train_mean,test_mean,train_missing_pct,test_missing_pct
0,core_temp,0.146000,4.661104e-179,59.229526,62.990345,0.000000,0.00000
1,sensor_25,0.144854,2.980530e-176,29.953079,32.096583,0.000000,0.00000
2,delta_ambient,0.139229,8.350716e-163,38.230592,41.952267,0.000000,0.00000
3,sensor_14,0.131042,3.369093e-144,-34.303976,-36.283095,0.000000,0.00000
4,sensor_06,0.107125,2.169975e-96,33.193240,32.204642,0.000000,0.00000
5,harmonic_thd,0.091854,6.171524e-71,3.205092,3.480921,0.000000,0.00000
6,grid_freq,0.083271,2.342515e-58,49.900486,49.924043,0.000000,0.00000
7,load_factor,0.082083,1.028762e-56,6.566271,7.253202,0.000000,0.00000
8,humidity,0.079465,5.715894e-49,47.733197,50.443544,7.891667,8.04375
9,sensor_09,0.075188,1.208656e-47,107.857737,104.272153,0.000000,0.00000


## 4. Exact 5-fold OOF baseline

This is the main checkpoint. The same fold-built 58-feature tables are reused for CatBoost and LightGBM, avoiding duplicate preprocessing work.

Expected reference results from the successful experiment:

- CatBoost OOF: approximately `0.190136`
- LightGBM OOF: approximately `0.196971`
- fixed 80/20 blend OOF: approximately `0.189636`


In [11]:
fixed_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

fixed_folds = list(fixed_cv.split(X_raw))

print("Number of fixed folds:", len(fixed_folds))


Number of fixed folds: 5


In [12]:
def create_cat_model():
    return CatBoostRegressor(
        iterations=2000,
        depth=6,
        learning_rate=0.03,
        l2_leaf_reg=3.0,
        loss_function="RMSE",
        random_seed=SEED,
        verbose=0,
        early_stopping_rounds=100,
    )


def create_lgb_model():
    return LGBMRegressor(
        objective="regression",
        n_estimators=4000,
        learning_rate=0.02,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=30,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    )


In [13]:
cat_oof_log = np.zeros(len(X_raw))
lgb_oof_log = np.zeros(len(X_raw))

cat_fold_scores = []
lgb_fold_scores = []

cat_best_iterations = []
lgb_best_iterations = []

for fold, (train_idx, valid_idx) in enumerate(
    fixed_folds,
    start=1,
):
    print("\n" + "=" * 60)
    print(f"BASELINE FOLD {fold}")
    print("=" * 60)

    X_train_fold_raw = X_raw.iloc[train_idx].copy()
    X_valid_fold_raw = X_raw.iloc[valid_idx].copy()

    y_train_fold = y_log.iloc[train_idx]
    y_valid_fold = y_log.iloc[valid_idx]

    (
        X_train_fold,
        X_valid_fold,
        _,
        _,
    ) = build_fold_features(
        X_train_fold_raw,
        X_valid_fold_raw,
    )

    print("Train shape:", X_train_fold.shape)
    print("Validation shape:", X_valid_fold.shape)

    # CatBoost
    cat_model = create_cat_model()

    cat_model.fit(
        X_train_fold,
        y_train_fold,
        eval_set=(X_valid_fold, y_valid_fold),
        use_best_model=True,
        verbose=False,
    )

    cat_fold_log = cat_model.predict(X_valid_fold)
    cat_oof_log[valid_idx] = cat_fold_log

    cat_fold_pred = np.clip(np.expm1(cat_fold_log), 0, None)
    fold_actual = np.expm1(y_valid_fold)

    cat_score = calculate_rmsle(
        fold_actual,
        cat_fold_pred,
    )

    cat_fold_scores.append(cat_score)
    cat_best_iterations.append(cat_model.get_best_iteration())

    # LightGBM
    lgb_model = create_lgb_model()

    lgb_model.fit(
        X_train_fold,
        y_train_fold,
        eval_set=[(X_valid_fold, y_valid_fold)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    lgb_fold_log = lgb_model.predict(
        X_valid_fold,
        num_iteration=lgb_model.best_iteration_,
    )
    lgb_oof_log[valid_idx] = lgb_fold_log

    lgb_fold_pred = np.clip(np.expm1(lgb_fold_log), 0, None)

    lgb_score = calculate_rmsle(
        fold_actual,
        lgb_fold_pred,
    )

    lgb_fold_scores.append(lgb_score)
    lgb_best_iterations.append(lgb_model.best_iteration_)

    print("CatBoost RMSLE:", cat_score)
    print("CatBoost best iteration:", cat_model.get_best_iteration())
    print("LightGBM RMSLE:", lgb_score)
    print("LightGBM best iteration:", lgb_model.best_iteration_)



BASELINE FOLD 1
Train shape: (19200, 58)
Validation shape: (4800, 58)
CatBoost RMSLE: 0.19377168144710952
CatBoost best iteration: 1930
LightGBM RMSLE: 0.2023844467128128
LightGBM best iteration: 1319

BASELINE FOLD 2
Train shape: (19200, 58)
Validation shape: (4800, 58)
CatBoost RMSLE: 0.18855780787060383
CatBoost best iteration: 1557
LightGBM RMSLE: 0.19472330708843621
LightGBM best iteration: 1483

BASELINE FOLD 3
Train shape: (19200, 58)
Validation shape: (4800, 58)
CatBoost RMSLE: 0.1884482802589682
CatBoost best iteration: 1520
LightGBM RMSLE: 0.19328720840093425
LightGBM best iteration: 1749

BASELINE FOLD 4
Train shape: (19200, 58)
Validation shape: (4800, 58)
CatBoost RMSLE: 0.1879147839158609
CatBoost best iteration: 1593
LightGBM RMSLE: 0.19577980129277028
LightGBM best iteration: 1396

BASELINE FOLD 5
Train shape: (19200, 58)
Validation shape: (4800, 58)
CatBoost RMSLE: 0.19191749463866725
CatBoost best iteration: 1567
LightGBM RMSLE: 0.19855074861966412
LightGBM best iter

In [14]:
cat_oof_pred = np.clip(np.expm1(cat_oof_log), 0, None)
lgb_oof_pred = np.clip(np.expm1(lgb_oof_log), 0, None)

blend_oof_log = (
    0.80 * cat_oof_log
    + 0.20 * lgb_oof_log
)

blend_oof_pred = np.clip(
    np.expm1(blend_oof_log),
    0,
    None,
)

cat_global_oof = calculate_rmsle(y_raw, cat_oof_pred)
lgb_global_oof = calculate_rmsle(y_raw, lgb_oof_pred)
blend_global_oof = calculate_rmsle(y_raw, blend_oof_pred)

print("CatBoost global OOF:", cat_global_oof)
print("LightGBM global OOF:", lgb_global_oof)
print("80/20 blend global OOF:", blend_global_oof)

print("\nCatBoost fold scores:", cat_fold_scores)
print("LightGBM fold scores:", lgb_fold_scores)

print("\nCatBoost best iterations:", cat_best_iterations)
print("LightGBM best iterations:", lgb_best_iterations)


CatBoost global OOF: 0.1901360336889869
LightGBM global OOF: 0.1969714222407914
80/20 blend global OOF: 0.18963602921932898

CatBoost fold scores: [0.19377168144710952, 0.18855780787060383, 0.1884482802589682, 0.1879147839158609, 0.19191749463866725]
LightGBM fold scores: [0.2023844467128128, 0.19472330708843621, 0.19328720840093425, 0.19577980129277028, 0.19855074861966412]

CatBoost best iterations: [1930, 1557, 1520, 1593, 1567]
LightGBM best iterations: [1319, 1483, 1749, 1396, 1032]


## 5. Robustness checkpoints

These metrics answer different questions:

- **Global OOF:** average performance across all training rows.
- **Test-like OOF:** performance on the 20% of training rows most similar to test.
- **Young-hot OOF:** performance in the known difficult region.
- **Fold spread:** stability across folds.

A candidate is worth considering only when its global OOF is competitive and it improves or preserves the test-like checkpoint.

In [15]:
young_threshold = train["asset_age"].quantile(0.25)
hot_threshold = train["core_temp"].quantile(0.75)

young_hot_mask = (
    (train["asset_age"] <= young_threshold)
    & (train["core_temp"] >= hot_threshold)
)

checkpoint_rows = []

for model_name, prediction in {
    "CatBoost": cat_oof_pred,
    "LightGBM": lgb_oof_pred,
    "Blend_80_20": blend_oof_pred,
}.items():
    checkpoint_rows.append({
        "model": model_name,
        "global_oof": calculate_rmsle(
            y_raw,
            prediction,
        ),
        "test_like_oof": calculate_rmsle(
            y_raw[very_test_like_mask],
            prediction[very_test_like_mask],
        ),
        "young_hot_oof": calculate_rmsle(
            y_raw[young_hot_mask],
            prediction[young_hot_mask],
        ),
    })

checkpoint_table = pd.DataFrame(checkpoint_rows)
display(checkpoint_table)


,model,global_oof,test_like_oof,young_hot_oof
0,CatBoost,0.190136,0.231128,0.288159
1,LightGBM,0.196971,0.238572,0.296824
2,Blend_80_20,0.189636,0.230337,0.287495


In [16]:
actual_log = np.log1p(y_raw.values)

cat_error = cat_oof_log - actual_log
lgb_error = lgb_oof_log - actual_log

error_correlation = np.corrcoef(
    cat_error,
    lgb_error,
)[0, 1]

fold_stability = pd.DataFrame({
    "fold": np.arange(1, 6),
    "cat_rmsle": cat_fold_scores,
    "lgb_rmsle": lgb_fold_scores,
})

print("CatBoost/LightGBM OOF error correlation:", error_correlation)
display(fold_stability)


CatBoost/LightGBM OOF error correlation: 0.9406231398448397


,fold,cat_rmsle,lgb_rmsle
0,1,0.193772,0.202384
1,2,0.188558,0.194723
2,3,0.188448,0.193287
3,4,0.187915,0.195780
4,5,0.191917,0.198551


### Optional blend diagnostic

This does not automatically replace the proven 80/20 blend. It shows whether a candidate model changes the preferred blend region.

In [17]:
blend_diagnostic = []

for cat_weight in np.arange(0.50, 1.01, 0.05):
    lgb_weight = 1.0 - cat_weight

    candidate_log = (
        cat_weight * cat_oof_log
        + lgb_weight * lgb_oof_log
    )

    candidate_pred = np.clip(
        np.expm1(candidate_log),
        0,
        None,
    )

    blend_diagnostic.append({
        "cat_weight": round(float(cat_weight), 2),
        "lgb_weight": round(float(lgb_weight), 2),
        "global_oof": calculate_rmsle(
            y_raw,
            candidate_pred,
        ),
        "test_like_oof": calculate_rmsle(
            y_raw[very_test_like_mask],
            candidate_pred[very_test_like_mask],
        ),
        "young_hot_oof": calculate_rmsle(
            y_raw[young_hot_mask],
            candidate_pred[young_hot_mask],
        ),
    })

blend_diagnostic = pd.DataFrame(blend_diagnostic)
display(blend_diagnostic.sort_values("global_oof"))


,cat_weight,lgb_weight,global_oof,test_like_oof,young_hot_oof
6,0.80,0.20,0.189636,0.230337,0.287495
5,0.75,0.25,0.189659,0.230320,0.287518
7,0.85,0.15,0.189672,0.230427,0.287547
4,0.70,0.30,0.189741,0.230374,0.287617
8,0.90,0.10,0.189768,0.230589,0.287675
3,0.65,0.35,0.189882,0.230501,0.287792
9,0.95,0.05,0.189922,0.230822,0.287880
2,0.60,0.40,0.190083,0.230700,0.288043
10,1.00,-0.00,0.190136,0.231128,0.288159
1,0.55,0.45,0.190342,0.230970,0.288369


## 6. Save the baseline checkpoint

This prevents future experiments from losing or silently replacing the proven baseline.

In [24]:
CHECKPOINT_DIR = (
    "/content/drive/MyDrive/"
    "EDI_Hackathon/baseline_checkpoint"
)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

np.save(
    os.path.join(CHECKPOINT_DIR, "cat_oof_log.npy"),
    cat_oof_log,
)
np.save(
    os.path.join(CHECKPOINT_DIR, "lgb_oof_log.npy"),
    lgb_oof_log,
)
np.save(
    os.path.join(CHECKPOINT_DIR, "blend_oof_log.npy"),
    blend_oof_log,
)
np.save(
    os.path.join(CHECKPOINT_DIR, "train_test_likeness.npy"),
    train_test_likeness,
)
np.save(
    os.path.join(CHECKPOINT_DIR, "very_test_like_mask.npy"),
    very_test_like_mask,
)

checkpoint_table.to_csv(
    os.path.join(CHECKPOINT_DIR, "checkpoint_scores.csv"),
    index=False,
)

baseline_metadata = {
    "seed": SEED,
    "feature_count": 58,
    "cat_global_oof": cat_global_oof,
    "lgb_global_oof": lgb_global_oof,
    "blend_global_oof": blend_global_oof,
    "shift_auc": float(shift_auc),
    "cat_best_iterations": [
        int(value) for value in cat_best_iterations
    ],
    "lgb_best_iterations": [
        int(value) for value in lgb_best_iterations
    ],
    "proven_kaggle_score": 0.20500,
    "proven_cat_final_iterations": 1840,
    "proven_lgb_final_iterations": 1396,
    "blend_cat_weight": 0.80,
    "blend_lgb_weight": 0.20,
}

with open(
    os.path.join(CHECKPOINT_DIR, "baseline_metadata.json"),
    "w",
) as file:
    json.dump(baseline_metadata, file, indent=2)

print("Saved checkpoint files to:", CHECKPOINT_DIR)


Saved checkpoint files to: /content/drive/MyDrive/EDI_Hackathon/baseline_checkpoint


In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
import shutil

shutil.make_archive(
    "/content/baseline_checkpoint",
    "zip",
    CHECKPOINT_DIR
)

print("ZIP created")

ZIP created


## 7. Candidate experiment scoreboard

For every future experiment:

1. produce fully out-of-fold log predictions;
2. evaluate them with this function;
3. compare against `Blend_80_20`;
4. reject candidates that improve only one tiny metric while harming robustness.

Do not create a submission from a candidate before this comparison.

In [19]:
BASELINE_GLOBAL = blend_global_oof
BASELINE_TEST_LIKE = calculate_rmsle(
    y_raw[very_test_like_mask],
    blend_oof_pred[very_test_like_mask],
)
BASELINE_YOUNG_HOT = calculate_rmsle(
    y_raw[young_hot_mask],
    blend_oof_pred[young_hot_mask],
)


def evaluate_candidate_oof(
    candidate_name,
    candidate_oof_log,
):
    candidate_oof_log = np.asarray(candidate_oof_log)

    if candidate_oof_log.shape != (len(train),):
        raise ValueError(
            f"Expected shape {(len(train),)}, "
            f"received {candidate_oof_log.shape}"
        )

    candidate_pred = np.clip(
        np.expm1(candidate_oof_log),
        0,
        None,
    )

    result = pd.DataFrame([{
        "candidate": candidate_name,
        "global_oof": calculate_rmsle(
            y_raw,
            candidate_pred,
        ),
        "global_change": calculate_rmsle(
            y_raw,
            candidate_pred,
        ) - BASELINE_GLOBAL,
        "test_like_oof": calculate_rmsle(
            y_raw[very_test_like_mask],
            candidate_pred[very_test_like_mask],
        ),
        "test_like_change": calculate_rmsle(
            y_raw[very_test_like_mask],
            candidate_pred[very_test_like_mask],
        ) - BASELINE_TEST_LIKE,
        "young_hot_oof": calculate_rmsle(
            y_raw[young_hot_mask],
            candidate_pred[young_hot_mask],
        ),
        "young_hot_change": calculate_rmsle(
            y_raw[young_hot_mask],
            candidate_pred[young_hot_mask],
        ) - BASELINE_YOUNG_HOT,
    }])

    return result


# Example:
# candidate_report = evaluate_candidate_oof(
#     "CatBoost_depth_5",
#     candidate_oof_log,
# )
# display(candidate_report)


### Practical decision rule

Lower RMSLE is better.

A candidate is **promising** when:

- global OOF improves by a non-trivial amount, or stays almost unchanged;
- test-like OOF clearly improves;
- no severe deterioration appears in young-hot or individual folds;
- the result comes from the same fixed folds and fold-safe preprocessing.

A difference around `0.00001` is normally too small to trust. Keep the exact numerical report rather than declaring a win from rounding.

## 8. Exact final model and submission

Run this only for the proven baseline or for a candidate that passed the checkpoints.

The fixed iterations below reproduce the successful 0.20500 pipeline:

- CatBoost: `1840`
- LightGBM: `1396`
- log-space blend: `80% / 20%`


In [20]:
X_full_train_thermal, X_full_test_thermal, final_temperature_model = (
    add_temperature_residual_features(
        X_raw.copy(),
        X_test_raw.copy(),
    )
)

X_full_train, X_full_test, full_medians = preprocess_fold(
    X_full_train_thermal,
    X_full_test_thermal,
)

X_full_train = add_engineered_features_stress(X_full_train)
X_full_test = add_engineered_features_stress(X_full_test)

assert list(X_full_train.columns) == list(X_full_test.columns)
assert X_full_train.shape == (24000, 58)
assert X_full_test.shape == (16000, 58)
assert X_full_train.isna().sum().sum() == 0
assert X_full_test.isna().sum().sum() == 0

print("Final train features:", X_full_train.shape)
print("Final test features:", X_full_test.shape)


Final train features: (24000, 58)
Final test features: (16000, 58)


In [21]:
FINAL_CAT_ITERATIONS = 1840
FINAL_LGB_ITERATIONS = 1396

final_cat_model = CatBoostRegressor(
    iterations=FINAL_CAT_ITERATIONS,
    depth=6,
    learning_rate=0.03,
    l2_leaf_reg=3.0,
    loss_function="RMSE",
    random_seed=SEED,
    verbose=100,
)

final_cat_model.fit(
    X_full_train,
    y_log,
)

cat_test_log = final_cat_model.predict(X_full_test)


0:	learn: 0.9283741	total: 27.7ms	remaining: 50.9s
100:	learn: 0.2744276	total: 2.42s	remaining: 41.7s
200:	learn: 0.2143198	total: 6.36s	remaining: 51.9s
300:	learn: 0.1983686	total: 9.02s	remaining: 46.1s
400:	learn: 0.1901752	total: 11.1s	remaining: 39.9s
500:	learn: 0.1843760	total: 13.2s	remaining: 35.3s
600:	learn: 0.1801740	total: 15.3s	remaining: 31.5s
700:	learn: 0.1767324	total: 17.4s	remaining: 28.3s
800:	learn: 0.1737308	total: 21.3s	remaining: 27.6s
900:	learn: 0.1710598	total: 24.1s	remaining: 25.1s
1000:	learn: 0.1685288	total: 26.2s	remaining: 22s
1100:	learn: 0.1661713	total: 28.3s	remaining: 19s
1200:	learn: 0.1638472	total: 30.4s	remaining: 16.2s
1300:	learn: 0.1616742	total: 32.6s	remaining: 13.5s
1400:	learn: 0.1595701	total: 36.1s	remaining: 11.3s
1500:	learn: 0.1575931	total: 39.3s	remaining: 8.87s
1600:	learn: 0.1555828	total: 41.4s	remaining: 6.18s
1700:	learn: 0.1537211	total: 43.5s	remaining: 3.56s
1800:	learn: 0.1518733	total: 45.7s	remaining: 990ms
1839:	le

In [22]:
final_lgb_model = LGBMRegressor(
    objective="regression",
    n_estimators=FINAL_LGB_ITERATIONS,
    learning_rate=0.02,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

final_lgb_model.fit(
    X_full_train,
    y_log,
)

lgb_test_log = final_lgb_model.predict(X_full_test)


In [23]:
final_blend_test_log = (
    0.80 * cat_test_log
    + 0.20 * lgb_test_log
)

final_blend_test_pred = np.clip(
    np.expm1(final_blend_test_log),
    0,
    None,
)

submission = pd.DataFrame({
    "id": test["id"].values,
    "edi": final_blend_test_pred,
})

submission.to_csv(
    "submission_clean_best_020500.csv",
    index=False,
)

print("Submission shape:", submission.shape)
print("Missing:", submission["edi"].isna().sum())
print("Infinite:", np.isinf(submission["edi"]).sum())
print("Duplicate IDs:", submission["id"].duplicated().sum())
print("Minimum prediction:", submission["edi"].min())
print("Maximum prediction:", submission["edi"].max())

display(submission.head())

assert submission.shape == (16000, 2)
assert submission["edi"].isna().sum() == 0
assert np.isfinite(submission["edi"]).all()
assert submission["id"].equals(test["id"])


Submission shape: (16000, 2)
Missing: 0
Infinite: 0
Duplicate IDs: 0
Minimum prediction: 3.9652839915435703
Maximum prediction: 737.3295358422752


,id,edi
0,24000,46.171586
1,24001,37.690626
2,24002,51.375100
3,24003,78.267724
4,24004,31.575184


## 9. What to work on next

The clean notebook establishes the compass. The next experiments should be controlled and limited to one hypothesis at a time.

Recommended order:

1. CatBoost robustness: depth and regularization.
2. Re-evaluate blend weights using both global and test-like checkpoints.
3. Improve the thermal residual predictor, because thermal residuals already helped the leaderboard.
4. Carefully test pseudo-labeling only after the validation checkpoints are stable.
5. Consider external or synthetic data only when its physical compatibility and labels are defensible.

Do not return to broad GMM clustering, manual regional corrections, or dozens of unmotivated interactions unless new evidence supports them.
